# Parent-query learning curve (one fixed seed)

This notebook is a minimally changed copy of `train_llama_seed.ipynb` for the
parent-query supervision-budget experiment.

- Run it once for each budget: **10, 25, 50, 75, and 100 parent queries**.
- The only experiment seed is **42**. This gives five training runs, not fifteen.
- Set `PARENT_BUDGET` in the environment before starting a clean kernel, or edit
  the default in the configuration cell. For example, in PowerShell:
  `$env:PARENT_BUDGET="25"`.
- Parent subsets are balanced across the five DORIS query categories and nested:
  every smaller-budget subset is contained in every larger one.
- All aspect rows belonging to a selected parent are retained. Pool-augmentation
  random seeds use the original row identity, so shared rows receive identical
  sampled pools at every budget.
- Each checkpoint contains `parent_query_manifest.json` with the exact selected
  parents and run settings.

Evaluate all five checkpoints with the same held-out evaluation pipeline. With
one seed, report the curve as evidence for this fixed run and do not claim that
seed-to-seed variability was measured.


In [ ]:
pip install transformers datasets peft trl bitsandbytes accelerate ipywidgets tensorboard optuna

In [ ]:
pip install unsloth

In [ ]:
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"

In [ ]:
import os, re
import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import random
import numpy as np
import torch
from transformers import set_seed

In [ ]:
MAX_TOKENS=16_384

In [ ]:
RUN_SEED = 42
PARENT_BUDGETS = (10, 25, 50, 75, 100)
PARENT_BUDGET = int(os.environ.get("PARENT_BUDGET", "100"))

if PARENT_BUDGET not in PARENT_BUDGETS:
    raise ValueError(
        f"PARENT_BUDGET must be one of {PARENT_BUDGETS}, got {PARENT_BUDGET}."
    )

print(f"Learning-curve run: budget={PARENT_BUDGET}, seed={RUN_SEED}")


In [ ]:
random.seed(RUN_SEED)
np.random.seed(RUN_SEED)
torch.manual_seed(RUN_SEED)
torch.cuda.manual_seed_all(RUN_SEED)
set_seed(RUN_SEED)

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "meta-llama/Llama-3.1-8B-Instruct",
    max_seq_length = MAX_TOKENS,
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    load_in_16bit = True,
    full_finetuning = False, # [NEW!] We have full finetuning now!
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 128,           # Larger = higher accuracy, but might overfit
    lora_alpha = 256,  # Recommended alpha == r at least
    lora_dropout = 0.05,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias = "none",
    random_state = RUN_SEED,
)

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

In [ ]:
from datasets import load_from_disk

dataset = load_from_disk("reranking_dataset_doris_mae")


In [ ]:
# Remove aspect rows that contain no positive documents, as in the original run.
dataset = dataset.filter(
    lambda example: any(score > 0 for score in example["candidate_scores"]),
    desc="Removing zero-positive queries",
)

# Preserve the original post-filter row identity before any budget subset is taken.
# Pool augmentation therefore receives the same random seed for a row at every budget.
dataset = dataset.map(
    lambda _example, index: {"source_row_id": int(index)},
    with_indices=True,
    desc="Recording stable source row IDs",
)

import hashlib
import random
from collections import Counter
from typing import Any, Iterable, Mapping, Sequence

def parent_query_id(candidate_pool: Sequence[Any]) -> str:
    """Return an order-invariant identifier for a parent query's candidate pool."""
    canonical_pool = "|".join(str(int(doc_id)) for doc_id in sorted(candidate_pool))
    return hashlib.sha256(canonical_pool.encode("utf-8")).hexdigest()

def build_parent_catalog(
    rows: Iterable[Mapping[str, Any]],
) -> dict[str, str]:
    """Map every parent-query identifier to its DORIS query category."""
    catalog: dict[str, str] = {}
    for row in rows:
        query_id = parent_query_id(row["candidate_pool"])
        query_type = str(row["original_query_type"])
        previous_type = catalog.setdefault(query_id, query_type)
        if previous_type != query_type:
            raise ValueError(
                f"Parent query {query_id} occurs under both "
                f"{previous_type!r} and {query_type!r}."
            )
    return catalog

def _category_seed(seed: int, query_type: str) -> int:
    digest = hashlib.sha256(f"{seed}||{query_type}".encode("utf-8")).hexdigest()
    return int(digest[:16], 16)

def select_parent_query_ids(
    rows: Iterable[Mapping[str, Any]],
    *,
    budget: int,
    seed: int,
    allowed_budgets: Sequence[int] = (10, 25, 50, 75, 100),
) -> tuple[str, ...]:
    """Select a deterministic, category-balanced, nested parent-query subset."""
    if budget not in allowed_budgets:
        raise ValueError(
            f"PARENT_BUDGET must be one of {tuple(allowed_budgets)}, got {budget}."
        )

    catalog = build_parent_catalog(rows)
    query_types = sorted(set(catalog.values()))
    if not query_types:
        raise ValueError("No parent queries were found.")
    if budget % len(query_types) != 0:
        raise ValueError(
            f"Budget {budget} must be divisible by the {len(query_types)} "
            "query categories to keep the subset balanced."
        )

    per_type = budget // len(query_types)
    selected: list[str] = []
    for query_type in query_types:
        category_ids = sorted(
            query_id
            for query_id, catalog_type in catalog.items()
            if catalog_type == query_type
        )
        if len(category_ids) < per_type:
            raise ValueError(
                f"Category {query_type!r} has only {len(category_ids)} parent "
                f"queries, but budget {budget} requires {per_type}."
            )

        rng = random.Random(_category_seed(seed, query_type))
        rng.shuffle(category_ids)
        selected.extend(category_ids[:per_type])

    return tuple(sorted(selected))

available_parent_catalog = build_parent_catalog(dataset)
selected_parent_ids = select_parent_query_ids(
    dataset,
    budget=PARENT_BUDGET,
    seed=RUN_SEED,
    allowed_budgets=PARENT_BUDGETS,
)
selected_parent_id_set = set(selected_parent_ids)

dataset = dataset.filter(
    lambda example: parent_query_id(example["candidate_pool"])
    in selected_parent_id_set,
    desc=f"Selecting {PARENT_BUDGET} parent queries",
    load_from_cache_file=False,
)

selected_type_counts = Counter(
    available_parent_catalog[query_id] for query_id in selected_parent_ids
)
selected_parent_manifest = {
    "experiment": "parent_query_learning_curve",
    "run_seed": RUN_SEED,
    "parent_budget": PARENT_BUDGET,
    "allowed_parent_budgets": list(PARENT_BUDGETS),
    "selection_method": "nested category-balanced sampling without replacement",
    "parent_id_definition": "SHA-256 of sorted candidate document IDs",
    "available_parent_queries": len(available_parent_catalog),
    "selected_parent_queries": len(selected_parent_ids),
    "selected_aspect_rows": len(dataset),
    "selected_type_counts": dict(sorted(selected_type_counts.items())),
    "selected_parents": [
        {
            "parent_query_id": query_id,
            "original_query_type": available_parent_catalog[query_id],
        }
        for query_id in selected_parent_ids
    ],
}

assert len(selected_parent_ids) == PARENT_BUDGET
assert len(set(selected_type_counts.values())) == 1

print(f"Available parent queries: {len(available_parent_catalog)}")
print(f"Selected parent queries: {len(selected_parent_ids)}")
print(f"Selected aspect rows: {len(dataset)}")
print(f"Selected parent queries by type: {dict(sorted(selected_type_counts.items()))}")


In [ ]:
dataset[100]

In [ ]:
import pandas as pd

corpus_df = pd.read_json("splits/Corpus.json")

In [ ]:
corpus_df.head()

In [ ]:
len(corpus_df)

In [ ]:
print("Creating abstract_id -> text lookup dictionary...")
abstract_lookup = {}
for _, row in corpus_df.iterrows():
    title = row['title'].replace("_", " ")
    text = f"Title: {title}\nAbstract: {row['original_abstract']}"
    abstract_lookup[int(row['abstract_id'])] = text

print(f"Created lookup for {len(abstract_lookup)} abstracts.")

In [ ]:
abstract_lookup[0]

In [ ]:
import random, hashlib
from typing import List, Tuple, Dict, Any

# -----------------------------
# CONFIG
# -----------------------------
N_VARIANTS       = 3     # AT MOST this many unique pools per query
TARGET_SIZE      = 50
MAX_POSITIVES    = 5
HARD_NEG_TOP     = 100    # set None to use ALL negatives
MAX_TOTAL_TRIES  = 3000  # per query; stop early if can't find more unique pools

# Uniqueness definition:
# - set-based (order-invariant): same 20 doc_ids in different order counts as DUPLICATE
def pool_signature(doc_ids: List[Any]) -> Tuple[int, ...]:
    return tuple(sorted(int(d) for d in doc_ids))

# If you ever want "unique-by-order" instead, use:
# def pool_signature(doc_ids): return tuple(int(d) for d in doc_ids)


def stable_seed(row_id: int, query_text: str) -> int:
    # row_id prevents collisions if query_text repeats
    h = hashlib.md5(f"{row_id}||{query_text}".encode("utf-8")).hexdigest()
    return int(h[:8], 16)


# -----------------------------
# PROMPT + TARGET CREATION
# -----------------------------
def create_single_conversation_entry(
    query_text: str,
    selected_docs: List[Tuple[Any, int]],
    abstract_lookup: Dict[int, str],
) -> Dict[str, Any]:
    """
    selected_docs: list of (doc_id, score), already shuffled in input order
    Returns dict with:
      - conversations
      - pool_k, scores_k
    """
    passages_str_list = []
    scores_k = []
    pool_k = []

    for i, (doc_id, score) in enumerate(selected_docs):
        pool_k.append(doc_id)
        scores_k.append(int(score))

        doc_text = abstract_lookup.get(int(doc_id), "Error: Document text not found.")
        doc_text = " ".join(doc_text.split()[:300])  # truncate to ~300 words
        # doc_text = " ".join(doc_text.split())  # truncate to ~300 words
        passages_str_list.append(f"[{i+1}] {doc_text}")

    passages_for_prompt = "\n\n---\n\n".join(passages_str_list)

    user_msg = (
        f"You are an expert academic paper reranker. "
        f"Your task is to re-order the given list of passages (from [1] to [{len(selected_docs)}]) "
        f"based on their relevance to the query. Respond with only the ranking and nothing else.\n\n"
        f"Example output for 8 passages:\n"
        f"[2] > [5] > [4] > [8] > [6] > [1] > [3] > [7]\n\n"
        f"Query: {query_text}\n\n"
        f"Passages:\n{passages_for_prompt}\n\n"
        f"Your ranking (most to least relevant):"
    )

    # repeated_user_message = user_msg + "\n\n" + user_msg

    # Ground truth ranking: score desc, tie broken by earlier index (stable)
    indexed_scores = [(i+1, s) for i, s in enumerate(scores_k)]
    sorted_indices = sorted(indexed_scores, key=lambda x: (x[1], -x[0]), reverse=True)
    target_str = " > ".join(f"[{idx}]" for idx, _ in sorted_indices)

    conversation = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": target_str},
    ]

    return {"conversations": conversation, "pool_k": pool_k, "scores_k": scores_k}


# -----------------------------
# UNIQUE POOL SAMPLING
# -----------------------------
def sample_unique_pools_for_query(
    example: Dict[str, Any],
    *,
    rng: random.Random,
    n_variants: int,
    target_size: int,
    max_positives: int,
    hard_neg_top: int | None,
    max_total_tries: int,
) -> List[List[Tuple[Any, int]]]:
    pool = example["candidate_pool"]
    scores = example["candidate_scores"]

    # Bucket
    neg  = [(d, int(s)) for d, s in zip(pool, scores) if int(s) == 0]
    pos1 = [(d, int(s)) for d, s in zip(pool, scores) if int(s) == 1]
    pos2 = [(d, int(s)) for d, s in zip(pool, scores) if int(s) >= 2]
    all_pos = pos2 + pos1
    all_docs = all_pos + neg

    if hard_neg_top is not None and len(neg) > hard_neg_top:
        neg = neg[:hard_neg_top]

    # If pool is tiny, we'll just return <=1 variant from what exists.
    if len(all_docs) == 0:
        return []

    # Try structured modes first, then keep sampling "any" for coverage
    modes = ["only_2", "only_1", "mixed", "any", "any", "any"]

    seen = set()
    unique_selected_docs: List[List[Tuple[Any, int]]] = []

    tries = 0
    while len(unique_selected_docs) < n_variants and tries < max_total_tries:
        mode = modes[tries % len(modes)]
        tries += 1

        # ---- choose positives
        positives: List[Tuple[Any, int]] = []
        if mode == "only_2":
            k = min(len(pos2), max_positives)
            positives = rng.sample(pos2, k) if k > 0 else []
        elif mode == "only_1":
            k = min(len(pos1), max_positives)
            positives = rng.sample(pos1, k) if k > 0 else []
        elif mode == "mixed":
            k2 = min(len(pos2), max_positives // 2)
            k1 = min(len(pos1), max_positives - k2)
            if k2 > 0: positives += rng.sample(pos2, k2)
            if k1 > 0: positives += rng.sample(pos1, k1)
        else:  # "any"
            k = min(len(all_pos), max_positives)
            positives = rng.sample(all_pos, k) if k > 0 else []

        # ---- choose negatives
        needed = target_size - len(positives)
        if needed < 0:
            positives = positives[:target_size]
            needed = 0

        negatives: List[Tuple[Any, int]] = []
        if needed > 0:
            kneg = min(len(neg), needed)
            negatives = rng.sample(neg, kneg) if kneg > 0 else []

        if not positives and all_pos and max_positives > 0:
            positives = [rng.choice(all_pos)]

        selected = positives + negatives

        # ---- fill up to target_size with remaining docs (pos or neg) not already included
        if len(selected) < target_size:
            used_ids = {d for d, _ in selected}
            remaining = [p for p in all_docs if p[0] not in used_ids]
            fill = remaining[: max(0, target_size - len(selected))]
            selected += fill

        # If still < target_size because the overall pool is too small, that's fine:
        # we'll produce a smaller list (but signature uniqueness still applies).
        doc_ids = [d for d, _ in selected]
        sig = pool_signature(doc_ids)

        if sig in seen:
            continue

        seen.add(sig)
        rng.shuffle(selected)  # IMPORTANT: avoid positional shortcut
        unique_selected_docs.append(selected)

    return unique_selected_docs


# -----------------------------
# DATASET MAP FUNCTION (EXPANDS ROWS)
# -----------------------------
def formatting_prompts_func_unique(examples, indices):
    """
    Requires dataset.map(..., batched=True, with_indices=True)
    Assumes these exist in scope:
      - tokenizer
      - abstract_lookup
    """
    out_text, out_pools, out_scores = [], [], []
    n = len(examples["query_text"])

    for i in range(n):
        ex = {k: examples[k][i] for k in examples}
        source_row_id = int(ex["source_row_id"])
        qtext = ex["query_text"]

        rng = random.Random(stable_seed(source_row_id, qtext))

        unique_pools = sample_unique_pools_for_query(
            ex,
            rng=rng,
            n_variants=N_VARIANTS,
            target_size=TARGET_SIZE,
            max_positives=MAX_POSITIVES,
            hard_neg_top=HARD_NEG_TOP,
            max_total_tries=MAX_TOTAL_TRIES,
        )

        for selected_docs in unique_pools:
            processed = create_single_conversation_entry(
                query_text=qtext,
                selected_docs=selected_docs,
                abstract_lookup=abstract_lookup,
            )

            formatted = tokenizer.apply_chat_template(
                processed["conversations"],
                tokenize=False,
                add_generation_prompt=False,
            ).removeprefix("<bos>")

            out_text.append(formatted)
            out_pools.append(processed["pool_k"])
            out_scores.append(processed["scores_k"])

    return {"text": out_text, "candidate_pool_k": out_pools, "candidate_scores_k": out_scores}


# def has_positives(example):
#     return any(int(s) > 0 for s in example["candidate_scores"])

# dataset_filtered = dataset.filter(has_positives)

# print(f"Original: {len(dataset)}, After filtering: {len(dataset_filtered)}")
# print(f"Removed {len(dataset) - len(dataset_filtered)} queries with zero positives")

augmented_dataset = dataset.map(
    formatting_prompts_func_unique,
    batched=True,
    with_indices=True,
    remove_columns=dataset.column_names,
    load_from_cache_file=False,
)

print("Original size:", len(dataset))
print("Augmented size:", len(augmented_dataset))


In [ ]:
augmented_dataset[1]

In [ ]:
augmented_dataset[1]['text']


In [ ]:
bad_items = [
    (i, item)
    for i, item in enumerate(augmented_dataset)
    if not any(score > 0 for score in item["candidate_scores_k"])
]

print(f"Number of queries with zero positives: {len(bad_items)}\n")

for i, item in bad_items:
    print(f"Index: {i}")
    print(f"Query: {item.get('query_text', '<no query_text>')}")
    print(f"Candidate scores: {item['candidate_scores_k']}")
    print("-" * 80)


In [ ]:
import json

selected_parent_manifest["augmentation"] = {
    "n_variants": N_VARIANTS,
    "target_size": TARGET_SIZE,
    "max_positives": MAX_POSITIVES,
    "hard_negative_top": HARD_NEG_TOP,
    "augmented_training_examples": len(augmented_dataset),
}

print(json.dumps(selected_parent_manifest, indent=2))


In [ ]:
SAMPLE_INDEX = min(100, len(augmented_dataset) - 1)
augmented_dataset[SAMPLE_INDEX]


In [ ]:
print(
    f"Example of formatted training data:\n"
    f"{augmented_dataset[SAMPLE_INDEX]['text']}"
)


In [ ]:
def calculate_length(example):
    tokens = tokenizer(text=example["text"], truncation=False)
    return {"token_length": len(tokens["input_ids"])}


print("Calculating token lengths for the selected training set...")
dataset_with_lengths = augmented_dataset.map(calculate_length, num_proc=4)
max_tokens_in_set = max(dataset_with_lengths["token_length"])
print(f"The maximum number of tokens is: {max_tokens_in_set}")


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = augmented_dataset,
    eval_dataset = None,
    args = SFTConfig(
        max_seq_length = MAX_TOKENS,
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        warmup_steps = 20,
        # gradient_checkpointing = True,
        num_train_epochs = 2, # Set this for 1 full training run.
        # max_steps = 30,
        learning_rate = 1e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_torch",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = RUN_SEED,
        report_to = "none", # Use TrackIO/WandB etc
        bf16=True
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part    = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

In [ ]:
import json

from typing import Any, Iterable, Mapping

def summarize_training_tokens(
    rows: Iterable[Mapping[str, Any]],
    *,
    num_train_epochs: int,
) -> dict[str, int]:
    """Count post-tokenization input and supervised response tokens."""
    training_examples = 0
    input_tokens_per_epoch = 0
    supervised_response_tokens_per_epoch = 0
    max_sequence_tokens_after_truncation = 0

    for row in rows:
        input_ids = row["input_ids"]
        labels = row["labels"]
        sequence_tokens = len(input_ids)

        training_examples += 1
        input_tokens_per_epoch += sequence_tokens
        supervised_response_tokens_per_epoch += sum(
            int(label) != -100 for label in labels
        )
        max_sequence_tokens_after_truncation = max(
            max_sequence_tokens_after_truncation,
            sequence_tokens,
        )

    return {
        "training_examples": training_examples,
        "num_train_epochs": num_train_epochs,
        "input_tokens_per_epoch": input_tokens_per_epoch,
        "input_tokens_all_epochs": input_tokens_per_epoch * num_train_epochs,
        "supervised_response_tokens_per_epoch": (
            supervised_response_tokens_per_epoch
        ),
        "supervised_response_tokens_all_epochs": (
            supervised_response_tokens_per_epoch * num_train_epochs
        ),
        "max_sequence_tokens_after_truncation": (
            max_sequence_tokens_after_truncation
        ),
    }

def _json_safe_value(value: Any) -> Any:
    """Convert trainer metrics to values accepted by ``json.dumps``."""
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Mapping):
        return {
            str(key): _json_safe_value(item)
            for key, item in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [_json_safe_value(item) for item in value]
    if hasattr(value, "item"):
        return _json_safe_value(value.item())
    return str(value)

def summarize_trainer_run(
    trainer: Any,
    metrics: Mapping[str, Any],
) -> dict[str, Any]:
    """Capture actual optimizer progress and JSON-safe trainer metrics."""
    per_device_batch_size = int(trainer.args.per_device_train_batch_size)
    gradient_accumulation_steps = int(
        trainer.args.gradient_accumulation_steps
    )
    world_size = int(getattr(trainer.args, "world_size", 1))

    return {
        "optimizer_steps": int(trainer.state.global_step),
        "completed_epochs": _json_safe_value(trainer.state.epoch),
        "world_size": world_size,
        "per_device_train_batch_size": per_device_batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "effective_train_batch_size": (
            per_device_batch_size * gradient_accumulation_steps * world_size
        ),
        "metrics": _json_safe_value(metrics),
    }

selected_parent_manifest["token_accounting"] = summarize_training_tokens(
    trainer.train_dataset,
    num_train_epochs=int(trainer.args.num_train_epochs),
)
print(json.dumps(selected_parent_manifest["token_accounting"], indent=2))


In [ ]:
tokenizer.decode(trainer.train_dataset[SAMPLE_INDEX]["input_ids"])


In [ ]:
tokenizer.decode([
    tokenizer.pad_token_id if token_id == -100 else token_id
    for token_id in trainer.train_dataset[SAMPLE_INDEX]["labels"]
]).replace(tokenizer.pad_token, " ")


In [ ]:
trainer_stats = trainer.train()

selected_parent_manifest["training_results"] = summarize_trainer_run(
    trainer,
    trainer_stats.metrics,
)
print(json.dumps(selected_parent_manifest["training_results"], indent=2))


In [ ]:
from datetime import datetime
import json
from pathlib import Path

base_model_name = "Llama-3.1-8B-Instruct"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_directory = Path(
    f"models/{base_model_name}_variants_{N_VARIANTS}_target_{TARGET_SIZE}"
    f"_pos_{MAX_POSITIVES}_parent_budget_{PARENT_BUDGET}_seed_{RUN_SEED}/{timestamp}"
)

model.save_pretrained_merged(
    str(run_directory),
    tokenizer,
    save_method="merged_16bit",
)

selected_parent_manifest["checkpoint_directory"] = str(run_directory)
selected_parent_manifest["training"] = {
    "base_model": "meta-llama/Llama-3.1-8B-Instruct",
    "epochs": 2,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 1,
    "learning_rate": 1e-5,
    "warmup_steps": 20,
    "scheduler": "cosine",
}
manifest_path = run_directory / "parent_query_manifest.json"
manifest_path.write_text(
    json.dumps(selected_parent_manifest, indent=2) + "\n",
    encoding="utf-8",
)
print(f"Saved parent-query manifest to {manifest_path}")


In [ ]:
# import json

# bad = []
# for k, v in model.config.to_dict().items():
#     try:
#         json.dumps(v)
#     except TypeError:
#         bad.append((k, type(v), v))
# bad[:20], len(bad)


In [ ]:
# qc = model.config.quantization_config

# # defensive: qc might be a dict or a BitsAndBytesConfig-like object
# if isinstance(qc, dict):
#     qc.pop("get_loading_attributes", None)
# else:
#     # if it's an object, remove the attribute if present
#     if hasattr(qc, "get_loading_attributes"):
#         delattr(qc, "get_loading_attributes")

# # also: safest to ensure it's plain-JSON-able
# model.config.quantization_config = qc

# from datetime import datetime
# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# model.save_pretrained_merged(f"models/{timestamp}/gemma-3-finetune", tokenizer)


In [ ]:
# model.save_pretrained(f"models/{timestamp}/llama_varaints_3_target_size_50_positives_1")  # Local saving
# tokenizer.save_pretrained(f"models/{timestamp}/llama_varaints_3_target_size_50_positives_1")